# Chapter 2 Project — Titanic Survival Prediction

This project applies everything from chapter 2 on one real dataset — end to end.

**Goal:** Predict whether a passenger survived the Titanic disaster.

**What this covers:**
- EDA (2.6)
- Handling missing values (2.1)
- Handling outliers (2.2)
- Encoding categorical data (2.3)
- Feature engineering (2.5)
- Feature scaling (2.4)
- Training a baseline model and evaluating it

**Dataset:** Titanic passenger data — 891 rows, 12 columns

In [1]:
import pandas as pd
print("pandas ok")
import numpy as np
print("numpy ok")
import matplotlib.pyplot as plt
print("matplot ok")
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = sns.load_dataset('titanic')
print("Shape:", df.shape)
df.head()

pandas ok
numpy ok
matplot ok
Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## Step 1 — EDA

Understand the data before touching it.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [3]:
# Missing values
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
survived         0
pclass           0
sex              0
sibsp            0
parch            0
fare             0
class            0
who              0
adult_male       0
alive            0
alone            0
dtype: int64

In [4]:
# Class balance
print(df['survived'].value_counts(normalize=True).round(3) * 100)

survived
0    61.6
1    38.4
Name: proportion, dtype: float64


In [5]:
# Key groupby findings
print("Survival by sex:")
print(df.groupby('sex')['survived'].mean().round(2))

print("\nSurvival by pclass:")
print(df.groupby('pclass')['survived'].mean().round(2))

Survival by sex:
sex
female    0.74
male      0.19
Name: survived, dtype: float64

Survival by pclass:
pclass
1    0.63
2    0.47
3    0.24
Name: survived, dtype: float64


## Step 2 — Select & Clean Columns

Drop columns that are too messy or not useful for prediction.

In [6]:
# Keep only useful columns
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()

print("Working with:", df.shape)
df.head()

Working with: (891, 8)


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## Step 3 — Handle Missing Values (2.1)

In [7]:
# Age — fill with median (skewed distribution)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked — fill with mode (only 2 missing)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Verify
print("Missing values remaining:")
print(df.isnull().sum())

Missing values remaining:
survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


## Step 4 — Handle Outliers (2.2)

In [8]:
# Fare has extreme outliers — cap using IQR
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

print(f"Fare upper bound: {upper:.1f}")
print(f"Values above bound: {(df['fare'] > upper).sum()}")

df['fare'] = df['fare'].clip(upper=upper)
print(f"Fare max after capping: {df['fare'].max():.1f}")

Fare upper bound: 65.6
Values above bound: 116
Fare max after capping: 65.6


## Step 5 — Feature Engineering (2.5)

In [9]:
# family_size — travelling alone vs with family
df['family_size'] = df['sibsp'] + df['parch'] + 1

# is_alone — binary flag
df['is_alone'] = (df['family_size'] == 1).astype(int)

print("Survival rate by is_alone:")
print(df.groupby('is_alone')['survived'].mean().round(2))

df[['family_size', 'is_alone', 'survived']].head()

Survival rate by is_alone:
is_alone
0    0.51
1    0.30
Name: survived, dtype: float64


,family_size,is_alone,survived
0,2,0,0
1,2,0,1
2,1,1,1
3,2,0,1
4,1,1,0


## Step 6 — Encode Categorical Columns (2.3)

In [11]:
# sex — label encode (binary: male=1, female=0)
df['sex'] = LabelEncoder().fit_transform(df['sex'])

# embarked — one-hot encode (no natural order)
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

print("Columns after encoding:", df.columns.tolist())
df.head()

Columns after encoding: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'family_size', 'is_alone', 'embarked_Q', 'embarked_S']


,survived,pclass,sex,age,sibsp,parch,fare,family_size,is_alone,embarked_Q,embarked_S
0,0,3,1,22.0,1,0,7.2500,2,0,False,True
1,1,1,0,38.0,1,0,65.6344,2,0,False,False
2,1,3,0,26.0,0,0,7.9250,1,1,False,True
3,1,1,0,35.0,1,0,53.1000,2,0,False,True
4,0,3,1,35.0,0,0,8.0500,1,1,False,True


## Step 7 — Split, Scale, Train (2.4)

In [12]:
X = df.drop(columns=['survived'])
y = df['survived']

# Split first — always
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale after split — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

Train size: (712, 10)
Test size:  (179, 10)


## Step 8 — Train a Baseline Model

In [13]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.810

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84       105
           1       0.79      0.73      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



## Step 9 — Reflect

**Model:** Logistic Regression
**Accuracy:** 81%

**What the data needed:**
- Age had ~20% missing values → filled with median
- Embarked had 2 missing → filled with mode
- Fare had extreme outliers → capped with IQR
- Sex and embarked needed encoding
- Created family_size and is_alone as new features

**Key findings from EDA:**
- Women survived 74% vs men 19% → sex was the strongest feature
- 1st class survived 63% vs 3rd class 24% → pclass matters a lot
- Travelling alone had lower survival rate

**What could improve it:**
- Try Random Forest, KNN, SVM (chapter 4)
- Extract title from passenger name (Mr, Mrs, Miss)
- Handle class imbalance properly (chapter 7)

In [14]:
# Feature importance — which features did the model rely on most?
importance = pd.Series(
    abs(model.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

print("Feature importance (logistic regression coefficients):")
print(importance.round(3))

Feature importance (logistic regression coefficients):
sex            1.250
pclass         0.664
age            0.396
sibsp          0.376
is_alone       0.304
family_size    0.301
fare           0.210
embarked_S     0.155
parch          0.081
embarked_Q     0.023
dtype: float64
